# DNALLM MCP Client with LangChain Agents

This notebook demonstrates how to integrate DNALLM (DNA Large Language Model Toolkit) with LangChain agents using the Model Context Protocol (MCP). 

## Overview

This example shows how to:
1. **Start a DNALLM MCP server** with streamable HTTP transport
2. **Connect a LangChain client** to the MCP server
3. **Create an agent** that can use DNALLM's DNA analysis tools
4. **Perform comprehensive DNA sequence analysis** using multiple specialized models

## Key Components

- **DNALLM MCP Server**: Provides DNA analysis tools via MCP protocol
- **LangChain MCP Adapters**: Enables LangChain to communicate with MCP servers
- **Ollama Integration**: Uses local LLM (Qwen3) for natural language processing
- **Multi-Model Analysis**: Combines promoter, conservation, and chromatin analysis

## Prerequisites

- DNALLM installed and configured
- Ollama running with Qwen3 model
- Required Python packages installed (see first cell)

## Workflow

1. Install dependencies
2. Start DNALLM MCP server
3. Configure async environment
4. Initialize MCP client connection
5. Create LangChain agent with MCP tools
6. Analyze DNA sequence using the agent
7. Display comprehensive results


In [1]:
# Install required dependencies for LangChain MCP integration
# This cell installs the necessary packages to use LangChain with MCP (Model Context Protocol)
# and Ollama for local LLM inference

!uv pip install -U langchain                    # Core LangChain framework
!uv pip install -U langchain-mcp-adapters       # MCP adapters for LangChain integration
!uv pip install -U langchain-ollama            # Ollama integration for local LLM inference


Using Python 3.13.15 environment at: /home/forrest/Github/DNALLM/.scratch/mcp-example-venvs/langchain
⠙ Resolving dependencies...                                                     

⠹ Resolving dependencies...                                                     

⠸ Resolving dependencies...                                                     

⠸ langchain==1.4.3                                                              

⠼ langchain==1.4.3                                                              

...

⠧ pydantic==2.13.5                                                              

⠧ httpx2==2.13.1                                                                

⠇ httpx2==2.13.1                                                                

⠇ websockets==17.1                                                              

Resolved 35 packages in 1.61s
Checked 35 packages in 0.21ms


In [2]:
# Ensure the DNALLM MCP server is up (streamable HTTP transport)
# The server provides DNALLM's DNA analysis tools at MCP_URL below.
# A 2xx OR 4xx answer both prove a server is bound there: the MCP streamable-http
# endpoint answers a bare GET with 4xx by session/method semantics.
# If the server is already running (e.g. started in a terminal), this cell only detects it.
# Terminal alternative:  !dnallm mcp-server --transport streamable-http

import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request

MCP_HOST = "127.0.0.1"
MCP_PORT = 8000
MCP_URL = f"http://localhost:{MCP_PORT}/mcp"


def _mcp_up(timeout_s: float = 2.0) -> tuple[bool, str]:
    """Probe the MCP endpoint; any HTTP answer below 500 counts as up."""
    try:
        with urllib.request.urlopen(MCP_URL, timeout=timeout_s) as response:
            return True, f"HTTP {response.status}"
    except urllib.error.HTTPError as exc:
        return exc.code < 500, f"HTTP {exc.code}"
    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}"


_up, _evidence = _mcp_up()
if _up:
    print(f"DNALLM MCP server already running at {MCP_URL} ({_evidence})")
else:
    server_bin = shutil.which("dnallm-mcp-server")
    cmd = [server_bin] if server_bin else [sys.executable, "-m", "dnallm.mcp.server"]
    cmd += ["--transport", "streamable-http", "--host", MCP_HOST, "--port", str(MCP_PORT)]
    print(f"Starting DNALLM MCP server detached: {' '.join(cmd)}")
    # The log handle must outlive this cell (detached server process writes to it)
    _server_log = open("dnallm_mcp_server.log", "w")
    mcp_server_proc = subprocess.Popen(cmd, stdout=_server_log, stderr=_server_log, start_new_session=True)
    _deadline = time.time() + 120
    while time.time() < _deadline:
        _up, _evidence = _mcp_up()
        if _up:
            print(f"DNALLM MCP server ready at {MCP_URL} ({_evidence}); models lazy-load on first use")
            break
        time.sleep(2.0)
    else:
        raise RuntimeError(
            f"DNALLM MCP server did not become ready at {MCP_URL} within 120s; "
            f"see dnallm_mcp_server.log in the notebook directory (last probe: {_evidence})"
        )


DNALLM MCP server already running at http://localhost:8000/mcp (HTTP 406)


In [3]:
# Configure asyncio for Jupyter notebook compatibility
# nest_asyncio allows nested event loops, which is necessary for running async code in Jupyter
import nest_asyncio
import asyncio
import time

# Apply nest_asyncio to enable nested event loops
nest_asyncio.apply()

In [4]:
# Initialize MCP client to connect to DNALLM server
# This creates a connection to the DNALLM MCP server running on localhost:8000
from langchain_mcp_adapters.client import MultiServerMCPClient  
from langchain.agents import create_agent

# Create MCP client with DNALLM server configuration
# The server should be running on port 8000 with streamable HTTP transport
client = MultiServerMCPClient(  
    {
        "dnallm": {
            "transport": "streamable-http",  # HTTP-based remote server transport
            "url": "http://localhost:8000/mcp",  # DNALLM MCP server endpoint
        }
    }
)

In [5]:
# Create LangChain agent with MCP tools
# This retrieves available tools from the DNALLM MCP server and creates a LangChain agent
# that can use these tools for DNA sequence analysis

# Get available tools from the MCP server
tools = await client.get_tools()  

# Create a LangChain agent using Ollama's Qwen3 model with MCP tools
# The agent can now use DNALLM's DNA analysis capabilities through the MCP tools
agent = create_agent(
    "ollama:qwen3.8:latest",  # Local LLM model via Ollama
    tools                   # MCP tools from DNALLM server
)

In [6]:
# Perform DNA sequence analysis using the LangChain agent
# This demonstrates how to use the agent to analyze a DNA sequence using DNALLM's models
# The agent will automatically select and use the appropriate MCP tools for analysis

# Define the DNA sequence to analyze
dna_sequence = """AGAAAAAACATGACAAGAAATCGATAATAATACAAAAGCTATGATGGTGTGCAATGTCCGTGTGCATGCGTGCACGCATTGCAACCGGCCCAAATCAAGGCCCATCGATCAGTGAATACTCATGGGCCGGCGGCCCACCACCGCTTCATCTCCTCCTCCGACGACGGGAGCACCCCCGCCGCATCGCCACCGACGAGGAGGAGGCCATTGCCGGCGGCGCCCCCGGTGAGCCGCTGCACCACGTCCCTGA"""

# Invoke the agent to analyze the DNA sequence
# The agent will use DNALLM's specialized models to provide comprehensive analysis
dnallm_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": f'What is the function of following DNA sequence? Please analyze it thoroughly using all available models:\n{dna_sequence}'}]}
)

In [7]:
# Display the analysis results
# This prints the comprehensive DNA sequence analysis provided by the LangChain agent
# The analysis includes insights from multiple DNALLM models (promoter, conservation, chromatin)
# The results show detailed functional interpretation of the DNA sequence

print(dnallm_response['messages'][-1].content)

# DNA Sequence Functional Analysis

## Sequence profile
- **Length:** 154 bp
- **GC content:** ~42% (moderate)
- **Notable features:** a run of **polypyrimidine-like / poly-A** region at the 5′ end (`AGAAAAAA`), multiple **GC-rich dinucleotide clusters** toward the 3′ end (`...GCGGCCGG...GAGGAGGAGGCC...`), a **CCC-poly-A tract** (`...ACCCCCGCCG...`), and a poly-G/C-rich tail. No obvious Shine-Dalgarno, RBS, or strong consensus TATA-box signature is visually dominant.

## Multi-model predictions (all plant regulatory-element models)

| Model | Architecture | Prediction | Confidence |
|---|---|---|---|
| **Open chromatin** | DNAMamba (BPE) | **Full open** | **94.6%** (Not open 0.15%, Partial 5.3%) |
| **Promoter** | DNABERT (BPE) | **Not a core promoter** | 93.7% (Core promoter 6.3%) |
| **Conservation** | DNABERT (BPE) | **Not conserved** | 98.7% (Conserved 1.3%) |

## Integrated interpretation

Read together, these three signals tell a consistent story:

1. **Regulatory, not coding** —